#### Read all per-config agg500 metadata CSVs

#### Concatenate into a unified DataFrame

#### Stratify 5 folds by (alpha, q0) using sklearn

#### Save unified CSV + per-fold train/test CSVs

Cell 1 – Imports and paths

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# ------------------------------------------------------------------
# Adjust these to your setup
# ------------------------------------------------------------------
LOCAL_ROOT = "/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/"

# Root where the aggregated 500-jet dataset is stored
AGG_OUTPUT_ROOT = Path(
    f"{LOCAL_ROOT}ml-jet-ext/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500_20"
).expanduser()

AGG_METADATA_DIR = AGG_OUTPUT_ROOT / "metadata"

print("AGG_OUTPUT_ROOT:", AGG_OUTPUT_ROOT)
print("AGG_METADATA_DIR:", AGG_METADATA_DIR)
assert AGG_METADATA_DIR.exists(), "AGG_METADATA_DIR does not exist"


Cell 2 – Read all per-config agg500 metadata CSVs and concatenate

In [ ]:
# Find all per-config agg500 metadata CSVs
agg_meta_files = sorted(AGG_METADATA_DIR.glob("*_agg500_metadata.csv"))
print("Found metadata files:", len(agg_meta_files))
for p in agg_meta_files[:5]:
    print("  ", p.name)

if not agg_meta_files:
    raise FileNotFoundError(f"No *_agg500_metadata.csv files found in {AGG_METADATA_DIR}")


In [ ]:
# Load and concatenate all metadata
dfs = []
for p in agg_meta_files:
    df_cfg = pd.read_csv(p)
    dfs.append(df_cfg)

all_df = pd.concat(dfs, ignore_index=True)
print("Total aggregated samples:", len(all_df))
all_df.head()


Cell 3 – Sanity check that alpha and q0 columns exist

In [ ]:
assert "alpha" in all_df.columns, "'alpha' column missing in aggregated metadata"
assert "q0"   in all_df.columns, "'q0' column missing in aggregated metadata"

print("Unique alpha values:", sorted(all_df["alpha"].unique()))
print("Unique q0 values:", sorted(all_df["q0"].unique()))


Cell 4 – Build unified CSV (optional shuffle) before folds

In [ ]:
# Optional: shuffle once for good measure (StratifiedKFold will also shuffle)
all_df = all_df.sample(frac=1.0, random_state=0).reset_index(drop=True)

unified_csv = AGG_METADATA_DIR / "agg500_all.csv"
all_df.to_csv(unified_csv, index=False)
print(f"Saved unified (pre-fold) CSV to: {unified_csv}")


Cell 5 – Stratified 5-fold split by (alpha, q0) using sklearn

In [ ]:
N_FOLDS = 5
RANDOM_STATE = 0

# Build a combined label "alpha_q0" for stratification
strata = all_df[["alpha", "q0"]].astype(str).agg("_".join, axis=1)

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Initialize fold column
all_df["fold"] = -1

for fold_id, (_, test_idx) in enumerate(skf.split(X=all_df, y=strata)):
    all_df.loc[test_idx, "fold"] = fold_id

# Sanity check
assert (all_df["fold"] >= 0).all(), "Some rows did not get a fold assignment"

print("Fold counts:")
print(all_df["fold"].value_counts().sort_index())


Optional: inspect how (alpha, q0) is distributed across folds.

In [ ]:
alpha_q0_fold_counts = all_df.groupby(["fold", "alpha", "q0"]).size()
alpha_q0_fold_counts.unstack(level=0, fill_value=0)


Cell 6 – Save unified stratified CSV with fold assignments

In [ ]:
unified_strat_csv = AGG_METADATA_DIR / "agg500_all_stratified_alpha_q0.csv"
all_df.to_csv(unified_strat_csv, index=False)
print(f"Saved unified stratified CSV to: {unified_strat_csv}")


Cell 7 – Create per-fold train/test CSVs

In [ ]:
for f in range(N_FOLDS):
    fold_test = all_df[all_df["fold"] == f].reset_index(drop=True)
    fold_train = all_df[all_df["fold"] != f].reset_index(drop=True)

    train_csv = AGG_METADATA_DIR / f"agg500_alpha_q0_fold{f}_train.csv"
    test_csv  = AGG_METADATA_DIR / f"agg500_alpha_q0_fold{f}_test.csv"

    fold_train.to_csv(train_csv, index=False)
    fold_test.to_csv(test_csv, index=False)

    print(
        f"Fold {f}: train={len(fold_train)}, test={len(fold_test)} "
        f"-> {train_csv.name}, {test_csv.name}"
    )
